In [1]:
from utils import *
import numpy as np
import torch
from loguru import logger
import itertools

In [33]:
class IntegerComplex:
    def __init__(self, real: int, imag: int):
        if not isinstance(real, int) or not isinstance(imag, int):
            try:
                self.real = int(real)
                self.imag = int(imag)
            except ValueError:
                raise TypeError(
                    "IntegerComplex components must be convertible to integers."
                )
        else:
            self.real = real
            self.imag = imag

    def __repr__(self):
        return f"IntegerComplex({self.real}, {self.imag})"

    def __str__(self):
        if self.imag >= 0:
            return f"{self.real} + {self.imag}i"
        else:
            # Handle negative imaginary part cleanly
            return f"{self.real} - {-self.imag}i"

    def __add__(self, other):
        return IntegerComplex(self.real + other.real, self.imag + other.imag)

    def __sub__(self, other):
        return IntegerComplex(self.real - other.real, self.imag - other.imag)

    def __mul__(self, other):
        if isinstance(other, IntegerComplex):
            new_real = (self.real * other.real) - (self.imag * other.imag)
            new_imag = (self.real * other.imag) + (self.imag * other.real)
            return IntegerComplex(new_real, new_imag)
        elif isinstance(other, int):
            return IntegerComplex(other * self.real, other * self.imag)

    def __pow__(self, pow):
        assert isinstance(pow, int)
        if pow == 0:
            return IntegerComplex(1, 0)
        assert pow > 0

        rv = self
        for _ in range(pow - 1):
            rv *= self
        return rv

    def __eq__(self, other):
        assert isinstance(other, IntegerComplex)
        return self.real == other.real and self.imag == other.imag

    def __neg__(self):
        return IntegerComplex(-self.real, -self.imag)

    def conjugate(self):
        return IntegerComplex(self.real, -self.imag)

    def norm(self):
        return self.real**2 + self.imag**2


class Enumerator:
    def __init__(self, bound):
        self.bound = bound
        self.factor_base = {
            k: IntegerComplex(v[0], v[1])
            for k, v in get_all_decomposed_primes_up_to(self.bound).items()
            if k != 2
        }

    def get_all_points_from_factorization(self, factors):
        n_factors = len(factors)
        all_points = []
        for i in range(n_factors):
            g = self.factor_base[factors[i]]
            g_conj = g.conjugate()
            if len(all_points) > 0:
                all_points = [g * p for p in all_points] + [
                    g_conj * p for p in all_points
                ]
            else:
                all_points.append(g)
        return all_points

    def canonize_to_first_eighth(self, points):
        rv = set()
        for p in points:
            x = abs(p.real)
            y = abs(p.imag)
            rv.add((min(x, y), max(x, y)))
        return [IntegerComplex(v[0], v[1]) for v in rv]

    def meet_points(self, points):
        points = self.canonize_to_first_eighth(points)
        point_projs = [(p.real * p.imag) ** 2 for p in points]
        hashtable = dict()
        for i, p in enumerate(point_projs):
            for j, q in enumerate(point_projs[i:]):
                v = p + q
                if v in hashtable:
                    hashtable[v].append((i, j))
                else:
                    hashtable[v] = [(i, j)]
        return {k: v for k, v in hashtable.items() if len(v) > 1}

    def meet_points_from_factorization(self, factors, k=None, with_multiplicity=True):
        if k is None:
            return self.meet_points(self.get_all_points_from_factorization(factors))
        else:
            rv = []
            combinations = (
                itertools.combinations_with_replacement(factors, k)
                if with_multiplicity
                else itertools.combinations(factors, k)
            )
            for choice in tqdm.tqdm(list(combinations)):
                res = self.meet_points_from_factorization(choice)
                if len(res) > 0:
                    rv.append(res)
            return rv

    def meet_points_from_factor_base(self, k, with_multiplicity=True):
        return self.meet_points_from_factorization(
            list(self.factor_base.keys()), k, with_multiplicity=with_multiplicity
        )

In [34]:
BOUND = 1000

enum = Enumerator(BOUND)
len(enum.factor_base)

100%|██████████| 81/81 [00:00<00:00, 510885.15it/s]


80

In [35]:
enum.get_all_points_from_factorization([5, 13, 17, 61])

[IntegerComplex(-106, 237),
 IntegerComplex(178, 189),
 IntegerComplex(18, 259),
 IntegerComplex(246, 83),
 IntegerComplex(214, 147),
 IntegerComplex(218, -141),
 IntegerComplex(258, 29),
 IntegerComplex(126, -227)]

In [36]:
enum.meet_points_from_factorization([5, 13, 17, 61], 4)

100%|██████████| 35/35 [00:00<00:00, 49295.04it/s]


[{1762899048: [(1, 2), (2, 2)]}]

In [37]:
enum.meet_points_from_factor_base(4)

 15%|█▌        | 282831/1837620 [00:04<00:23, 66428.21it/s]


KeyboardInterrupt: 

100%|██████████| 1837620/1837620 [00:27<00:00, 67076.48it/s]


[{1762899048: [(1, 2), (2, 2)]},
 {86168670408: [(0, 2), (6, 1)]},
 {582449594625000: [(0, 2), (1, 4)]},
 {13369253110965000: [(1, 3), (3, 2)]}]

In [13]:
enum.factor_base

{5: IntegerComplex(2, 1),
 13: IntegerComplex(3, 2),
 17: IntegerComplex(4, 1),
 29: IntegerComplex(5, 2),
 37: IntegerComplex(6, 1),
 41: IntegerComplex(5, 4),
 53: IntegerComplex(7, 2),
 61: IntegerComplex(6, 5),
 73: IntegerComplex(8, 3),
 89: IntegerComplex(8, 5),
 97: IntegerComplex(9, 4),
 101: IntegerComplex(10, 1),
 109: IntegerComplex(10, 3),
 113: IntegerComplex(8, 7),
 137: IntegerComplex(11, 4),
 149: IntegerComplex(10, 7),
 157: IntegerComplex(11, 6),
 173: IntegerComplex(13, 2),
 181: IntegerComplex(10, 9),
 193: IntegerComplex(12, 7),
 197: IntegerComplex(14, 1),
 229: IntegerComplex(15, 2),
 233: IntegerComplex(13, 8),
 241: IntegerComplex(15, 4),
 257: IntegerComplex(16, 1),
 269: IntegerComplex(13, 10),
 277: IntegerComplex(14, 9),
 281: IntegerComplex(16, 5),
 293: IntegerComplex(17, 2),
 313: IntegerComplex(13, 12),
 317: IntegerComplex(14, 11),
 337: IntegerComplex(16, 9),
 349: IntegerComplex(18, 5),
 353: IntegerComplex(17, 8),
 373: IntegerComplex(18, 7),
 389: 